# PyTorch vs Keras: Flexibility Deep Dive
## Adult Census Income Classification (Dense + Dropout Only)

This notebook compares PyTorch and Keras frameworks using a **simple dense network with dropout** on the Adult Census Income dataset. We'll focus on **real flexibility differences** in:
- Custom loss functions
- Advanced training loops
- Model architecture modification
- Custom metrics implementation

**Dataset**: Adult Census Income (predict income >$50K)
**Architecture**: 3 dense layers + dropout only

*Run this in Google Colab for best experience*

# [https://tinyurl.com/CM3015MBonus1](https://tinyurl.com/CM3015MBonus1)

# Adult Census Income Dataset: Detailed Explanation

## Overview
The **Adult Census Income dataset** (also known as the "Census Income" dataset) is a classic machine learning dataset originally extracted from the 1994 US Census database by Barry Becker. It's widely used for binary classification tasks, particularly for predicting whether an individual's annual income exceeds $50,000 based on demographic and employment-related features.

**Source**: UCI Machine Learning Repository  
**Original URL**: https://archive.ics.uci.edu/ml/datasets/adult  
**Collection Year**: 1994 US Census  
**License**: Public domain (US Government data)

## Dataset Structure & Features

### Target Variable
- **`income`**: Binary target variable
  - `<=50K` = 0 (Annual income ≤ $50,000)
  - `>50K` = 1 (Annual income > $50,000)

### Feature Columns (14 total)

#### Numerical Features (6):
| Feature | Description | Data Type | Typical Range |
|---------|-------------|-----------|---------------|
| `age` | Age of the individual | Integer | 17-90 |
| `fnlwgt` | Final weight (number of people census believes entry represents) | Integer | 12,285-1,484,705 |
| `education_num` | Years of education (numerical representation) | Integer | 1-16 |
| `capital_gain` | Capital gains for the year | Integer | 0-99,999 |
| `capital_loss` | Capital losses for the year | Integer | 0-4,356 |
| `hours_per_week` | Hours worked per week | Integer | 1-99 |

#### Categorical Features (8):
| Feature | Description | Unique Values | Notable Categories |
|---------|-------------|---------------|-------------------|
| `workclass` | Employment sector | 9 | Private, Self-emp-not-inc, Local-gov, etc. |
| `education` | Highest education level | 16 | HS-grad, Bachelors, Masters, PhD, etc. |
| `marital_status` | Marital status | 7 | Married-civ-spouse, Never-married, Divorced, etc. |
| `occupation` | Job type | 15 | Prof-specialty, Craft-repair, Exec-managerial, etc. |
| `relationship` | Family role | 6 | Husband, Wife, Not-in-family, Own-child, etc. |
| `race` | Race/ethnicity | 5 | White, Black, Asian-Pac-Islander, Amer-Indian-Eskimo, Other |
| `sex` | Gender | 2 | Male, Female |
| `native_country` | Country of origin | 42 | United-States, Mexico, Philippines, Germany, etc. |

## Dataset Statistics & Characteristics

### Size & Shape
- **Total Records**: 48,842 (after removing missing values)
- **Training Set**: ~39,073 records (80%)
- **Test Set**: ~9,769 records (20%)
- **Features after encoding**: ~108 dimensions (after one-hot encoding)

### Class Distribution (Imbalanced Dataset)
| Class | Count | Percentage | Notes |
|-------|-------|------------|-------|
| `<=50K` | ~37,155 | ~76% | Majority class |
| `>50K` | ~11,687 | ~24% | Minority class |

**Key Challenge**: This dataset exhibits **class imbalance** (3:1 ratio), making it ideal for demonstrating:
- Custom loss functions (like focal loss)
- Threshold optimization
- Proper evaluation metrics beyond accuracy

### Missing Values & Data Quality
- **Missing Value Indicator**: `?` (question mark) in original dataset
- **Features with Missing Values**:
  - `workclass`: ~1,836 missing values
  - `occupation`: ~1,843 missing values
  - `native_country`: ~583 missing values
- **Handling**: Typically imputed with most frequent values or median for numerical features

## Why This Dataset is Perfect for Framework Comparison

### 1. **Real-World Complexity**
- Mix of numerical and categorical features
- Requires proper preprocessing (scaling, encoding, imputation)
- Class imbalance requires advanced techniques
- Realistic feature engineering challenges

### 2. **Ideal for Dense Network Demonstration**
- **After preprocessing**: ~108 features (perfect for dense layers)
- **Non-linear relationships**: Income prediction requires complex decision boundaries
- **Feature interactions**: Many features interact (e.g., education × occupation × age)

### 3. **Demonstrates Flexibility Requirements**
- **Custom loss functions**: Needed for class imbalance (focal loss)
- **Dynamic thresholding**: F1-score optimization requires threshold adjustment
- **Architecture modification**: Feature importance may require adaptive layer structures
- **Training control**: Gradient clipping needed due to varied feature scales

### 4. **Benchmarking Standard**
- Well-known baseline performance metrics
- Easy to compare results across frameworks
- Reproducible results due to stable dataset
- Common in research papers for framework comparisons

## Common Preprocessing Steps

```python
# Typical preprocessing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define feature groups
numeric_features = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
categorical_features = ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']

# Create preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
tf.random.set_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"TensorFlow/Keras version: {tf.__version__}")

PyTorch version: 2.9.0+cpu
TensorFlow/Keras version: 2.19.0


In [ ]:
# Download Adult Census Income dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

# Load data with proper handling
data = pd.read_csv(url, header=None, names=column_names, na_values=' ?', skipinitialspace=True)

# Preprocess target
data['income'] = data['income'].map({'<=50K': 0, '>50K': 1})

# Split features and target
X = data.drop('income', axis=1)
y = data['income']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {y.value_counts(normalize=True).to_dict()}")
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

Dataset shape: (32561, 14)
Class distribution: {0: 0.7591904425539756, 1: 0.2408095574460244}
Training samples: 26048, Test samples: 6513


In [ ]:
# Define preprocessing pipeline
numeric_features = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
categorical_features = ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed feature dimension: {X_train_processed.shape[1]}")

# Convert to PyTorch tensors
X_train_torch = torch.FloatTensor(X_train_processed)
y_train_torch = torch.LongTensor(y_train.values)
X_test_torch = torch.FloatTensor(X_test_processed)
y_test_torch = torch.LongTensor(y_test.values)

# Create PyTorch datasets
train_dataset = TensorDataset(X_train_torch, y_train_torch)
test_dataset = TensorDataset(X_test_torch, y_test_torch)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Processed feature dimension: 108


## Topic 1: Baseline Comparison
Same 3-layer dense network with dropout in both frameworks

In [ ]:
# Keras Baseline Model
input_dim = X_train_processed.shape[1]

keras_model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1, activation='sigmoid')
])

keras_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Train Keras model
keras_start_time = time.time()
keras_history = keras_model.fit(
    X_train_processed, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    verbose=1
)
keras_train_time = time.time() - keras_start_time

# Evaluate Keras model
keras_pred = keras_model.predict(X_test_processed, verbose=0).flatten()
keras_pred_binary = (keras_pred > 0.5).astype(int)

keras_acc = accuracy_score(y_test, keras_pred_binary)
keras_f1 = f1_score(y_test, keras_pred_binary)
keras_auc = roc_auc_score(y_test, keras_pred)

print(f"\nKeras Baseline Results:")
print(f"Training time: {keras_train_time:.2f} seconds")
print(f"Test Accuracy: {keras_acc:.4f}")
print(f"Test F1 Score: {keras_f1:.4f}")
print(f"Test AUC: {keras_auc:.4f}")

Epoch 1/10
326/326 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8122 - auc: 0.8313 - loss: 0.4027 - val_accuracy: 0.8470 - val_auc: 0.9054 - val_loss: 0.3275
Epoch 2/10
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8445 - auc: 0.9008 - loss: 0.3278 - val_accuracy: 0.8514 - val_auc: 0.9085 - val_loss: 0.3207
Epoch 3/10
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8503 - auc: 0.9039 - loss: 0.3242 - val_accuracy: 0.8518 - val_auc: 0.9085 - val_loss: 0.3203
Epoch 4/10
326/326 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8518 - auc: 0.9071 - loss: 0.3193 - val_accuracy: 0.8522 - val_auc: 0.9094 - val_loss: 0.3192
Epoch 5/10
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8542 - auc: 0.9079 - loss: 0.3180 - val_accuracy: 0.8536 - val_auc: 0.9095 - val_loss: 0.3189
Epoch 6/10
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8572 - auc: 0.9106 - loss: 0.3136 - val_accuracy: 0.8534 - val_auc: 0.9095 - val_loss: 0.3183
Epoch 7/10
326/326 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# PyTorch Baseline Model
class DenseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

pytorch_model = DenseNet(input_dim)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(pytorch_model.parameters(), lr=0.001)

# Training loop
pytorch_start_time = time.time()
epochs = 10

for epoch in range(epochs):
    pytorch_model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = pytorch_model(batch_x).squeeze()
        loss = criterion(outputs, batch_y.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

pytorch_train_time = time.time() - pytorch_start_time

# Evaluate PyTorch model
pytorch_model.eval()
with torch.no_grad():
    test_outputs = pytorch_model(X_test_torch).squeeze()
    pytorch_pred = torch.sigmoid(test_outputs).numpy()
    pytorch_pred_binary = (pytorch_pred > 0.5).astype(int)

pytorch_acc = accuracy_score(y_test, pytorch_pred_binary)
pytorch_f1 = f1_score(y_test, pytorch_pred_binary)
pytorch_auc = roc_auc_score(y_test, pytorch_pred)

print(f"\nPyTorch Baseline Results:")
print(f"Training time: {pytorch_train_time:.2f} seconds")
print(f"Test Accuracy: {pytorch_acc:.4f}")
print(f"Test F1 Score: {pytorch_f1:.4f}")
print(f"Test AUC: {pytorch_auc:.4f}")

Epoch 1/10, Loss: 0.3596
Epoch 2/10, Loss: 0.3157
Epoch 3/10, Loss: 0.3120
Epoch 4/10, Loss: 0.3091
Epoch 5/10, Loss: 0.3063
Epoch 6/10, Loss: 0.3027
Epoch 7/10, Loss: 0.3005
Epoch 8/10, Loss: 0.2995
Epoch 9/10, Loss: 0.2979
Epoch 10/10, Loss: 0.2959

PyTorch Baseline Results:
Training time: 12.30 seconds
Test Accuracy: 0.8641
Test F1 Score: 0.6987
Test AUC: 0.9173


## Topic 2: Advanced Flexibility Showcase: PyTorch vs Keras Deep Dive

This section demonstrates **real-world flexibility differences** between PyTorch and Keras frameworks when implementing advanced machine learning techniques. We focus on **practical scenarios** where framework choice significantly impacts development workflow and capabilities.

In [ ]:
# PyTorch: Custom Focal Loss Implementation
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        BCE_loss = nn.functional.binary_cross_entropy_with_logits(
            inputs, targets.float(), reduction='none'
        )
        pt = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        return focal_loss.mean()

# PyTorch: Custom F1 Score Metric
class F1Score:
    def __init__(self, threshold=0.5):
        self.threshold = threshold

    def __call__(self, predictions, targets):
        """Calculate F1 score for numpy arrays"""
        # Convert to numpy arrays if they're tensors
        if torch.is_tensor(predictions):
            predictions = predictions.cpu().numpy()
        if torch.is_tensor(targets):
            targets = targets.cpu().numpy()

        # Ensure proper data types
        predictions = predictions.astype(np.float32)
        targets = targets.astype(np.float32)

        # Apply threshold
        pred_binary = (predictions > self.threshold).astype(np.float32)

        # Calculate TP, FP, FN
        tp = np.sum(pred_binary * targets)
        fp = np.sum((1 - targets) * pred_binary)
        fn = np.sum(targets * (1 - pred_binary))

        # Calculate precision and recall
        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)

        # Calculate F1 score
        f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
        return f1

# PyTorch: Advanced Training Loop with Dynamic Features
class AdvancedPyTorchTrainer:
    def __init__(self, model, custom_loss, custom_metric, learning_rate=0.001):
        self.model = model
        #CUSTOM LOSS FUNCTION
        self.criterion = custom_loss
        #CUSTOM EVALUATION METRICS
        self.metric = custom_metric
        self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='max', factor=0.5, patience=2
        )
        self.verbose = True  # Manual verbose control

    def train(self, train_loader, val_loader, epochs=15):
        best_f1 = 0
        patience_counter = 0
        max_patience = 3
        best_model_state = None

        for epoch in range(epochs):
            self.model.train()
            total_loss = 0

            # Dynamic batch size adjustment based on epoch
            current_batch_size = min(64 + epoch * 8, 128)

            for batch_x, batch_y in train_loader:
                # Manual gradient clipping
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                self.optimizer.zero_grad()
                outputs = self.model(batch_x).squeeze()
                loss = self.criterion(outputs, batch_y)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)

            # Validation
            val_f1 = self.evaluate(val_loader)

            # Manual learning rate scheduling logging
            prev_lr = self.optimizer.param_groups[0]['lr']
            self.scheduler.step(val_f1)
            current_lr = self.optimizer.param_groups[0]['lr']

            if self.verbose:
                if current_lr < prev_lr:
                    print(f"  LR reduced from {prev_lr:.6f} to {current_lr:.6f}")

            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Val F1: {val_f1:.4f}")

            # Early stopping with custom logic
            if val_f1 > best_f1:
                best_f1 = val_f1
                patience_counter = 0
                best_model_state = self.model.state_dict()
            else:
                patience_counter += 1
                if patience_counter >= max_patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

        # Load best model
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
        return best_f1
    def evaluate(self, loader):
        self.model.eval()
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for batch_x, batch_y in loader:
                outputs = self.model(batch_x).squeeze()
                probs = torch.sigmoid(outputs)
                # Convert to numpy immediately
                all_preds.extend(probs.cpu().numpy())
                all_targets.extend(batch_y.cpu().numpy())

        # Now we have numpy arrays, call the metric
        return self.metric(np.array(all_preds), np.array(all_targets))

# Create validation set for PyTorch
X_train_val, X_val, y_train_val, y_val = train_test_split(
    X_train_processed, y_train, test_size=0.2, random_state=42, stratify=y_train
)

X_train_val_torch = torch.FloatTensor(X_train_val)
y_train_val_torch = torch.LongTensor(y_train_val.values)
X_val_torch = torch.FloatTensor(X_val)
y_val_torch = torch.LongTensor(y_val.values)

train_val_dataset = TensorDataset(X_train_val_torch, y_train_val_torch)
val_dataset = TensorDataset(X_val_torch, y_val_torch)

train_val_loader = DataLoader(train_val_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Train with advanced features
flexible_pytorch_model = DenseNet(input_dim)
focal_loss = FocalLoss(alpha=0.75, gamma=2.0)  # Higher alpha for class imbalance
f1_metric = F1Score(threshold=0.4)  # Optimize threshold for imbalanced data

trainer = AdvancedPyTorchTrainer(flexible_pytorch_model, focal_loss, f1_metric)
best_f1 = trainer.train(train_val_loader, val_loader, epochs=20)

print(f"\nPyTorch Advanced Training - Best Validation F1: {best_f1:.4f}")

Epoch 1/20, Loss: 0.0697, Val F1: 0.6660
Epoch 2/20, Loss: 0.0617, Val F1: 0.6869
Epoch 3/20, Loss: 0.0605, Val F1: 0.6702
Epoch 4/20, Loss: 0.0602, Val F1: 0.6765
  LR reduced from 0.001000 to 0.000500
Epoch 5/20, Loss: 0.0593, Val F1: 0.6869
Epoch 6/20, Loss: 0.0580, Val F1: 0.6754
Epoch 7/20, Loss: 0.0576, Val F1: 0.6792
  LR reduced from 0.000500 to 0.000250
Epoch 8/20, Loss: 0.0571, Val F1: 0.6726
Early stopping at epoch 8

PyTorch Advanced Training - Best Validation F1: 0.6869


In [ ]:
# Keras: Custom Focal Loss Implementation
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)

        BCE = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        pt = tf.exp(-BCE)
        focal_loss = alpha * tf.pow(1-pt, gamma) * BCE
        return tf.reduce_mean(focal_loss)
    return loss_fn

# Keras: Custom F1 Score Metric
class F1ScoreMetric(tf.keras.metrics.Metric):
    def __init__(self, threshold=0.5, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_binary = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)

        tp = tf.reduce_sum(y_true * y_pred_binary)
        fp = tf.reduce_sum((1 - y_true) * y_pred_binary)
        fn = tf.reduce_sum(y_true * (1 - y_pred_binary))

        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + 1e-7)
        recall = self.true_positives / (self.true_positives + self.false_negatives + 1e-7)
        return 2 * (precision * recall) / (precision + recall + 1e-7)

    def reset_state(self):
        self.true_positives.assign(0)
        self.false_positives.assign(0)
        self.false_negatives.assign(0)

# Keras: Advanced Training with Custom Callbacks
class DynamicTrainingCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best_f1 = 0
        self.patience_counter = 0
        self.max_patience = 3

    def _get_current_lr(self):
        """Safely get current learning rate across TF versions"""
        optimizer = self.model.optimizer
        try:
            # Modern TF 2.x approach
            if hasattr(optimizer, 'learning_rate'):
                lr_value = optimizer.learning_rate
                if hasattr(lr_value, 'numpy'):
                    return float(lr_value.numpy())
                return float(tf.keras.backend.get_value(lr_value))
        except:
            pass

        try:
            # Legacy TF 1.x/Keras approach
            if hasattr(optimizer, 'lr'):
                return float(tf.keras.backend.get_value(optimizer.lr))
        except:
            pass

        # Fallback to default
        return 0.001

    def _set_learning_rate(self, new_lr):
        """Safely set learning rate across TF versions"""
        optimizer = self.model.optimizer
        try:
            # Modern TF 2.x approach
            if hasattr(optimizer, 'learning_rate'):
                if hasattr(optimizer.learning_rate, 'assign'):
                    optimizer.learning_rate.assign(new_lr)
                    return True
        except:
            pass

        try:
            # Legacy TF 1.x/Keras approach
            if hasattr(optimizer, 'lr'):
                tf.keras.backend.set_value(optimizer.lr, new_lr)
                return True
        except:
            pass

        # Final fallback using backend
        try:
            tf.keras.backend.set_value(optimizer.learning_rate, new_lr)
            return True
        except:
            pass

        return False

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_f1 = logs.get('val_f1_score', 0)
        current_lr_value = self._get_current_lr()

        print(f"Epoch {epoch+1} - Val F1: {val_f1:.4f}, LR: {current_lr_value:.6f}")

        # Manual learning rate scheduling
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.patience_counter = 0
        else:
            self.patience_counter += 1
            if self.patience_counter >= 2:
                new_lr = current_lr_value * 0.5
                success = self._set_learning_rate(new_lr)
                if success:
                    print(f"  Reducing learning rate to {new_lr:.6f}")
                else:
                    print(f"  ⚠️ Failed to set LR, using {new_lr:.6f}")

        # Early stopping
        if self.patience_counter >= self.max_patience:
            self.model.stop_training = True
            print(f"  Early stopping triggered at epoch {epoch+1}")

# Keras model with custom loss and metrics
keras_flexible_model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1, activation='sigmoid')
])

# Compile with custom loss and metric
keras_flexible_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    # CUSTOM LOSS FUNCTION
    loss=focal_loss(gamma=2.0, alpha=0.75),
    # CUSTOM EVALUATION METRICS
    metrics=['accuracy', F1ScoreMetric(threshold=0.4), tf.keras.metrics.AUC(name='auc')]
)

# Callbacks setup
callbacks = [
    # CUSTOM CALLBACK FUNCTION
    DynamicTrainingCallback(),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_f1_score',
        patience=5,
        restore_best_weights=True,
        mode='max'
    )
]

# Train with custom callback
keras_flexible_history = keras_flexible_model.fit(
    X_train_val, y_train_val,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
# Evaluate Keras flexible model
keras_flexible_pred = keras_flexible_model.predict(X_test_processed, verbose=0).flatten()
keras_flexible_pred_binary = (keras_flexible_pred > 0.4).astype(int)

keras_flexible_acc = accuracy_score(y_test, keras_flexible_pred_binary)
keras_flexible_f1 = f1_score(y_test, keras_flexible_pred_binary)
keras_flexible_auc = roc_auc_score(y_test, keras_flexible_pred)

print(f"\nKeras Advanced Results:")
print(f"Test Accuracy: {keras_flexible_acc:.4f}")
print(f"Test F1 Score: {keras_flexible_f1:.4f}")
print(f"Test AUC: {keras_flexible_auc:.4f}")

Epoch 1/20
317/326 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7745 - auc: 0.7935 - f1_score: 0.5588 - loss: 0.0843Epoch 1 - Val F1: 0.6574, LR: 0.001000
326/326 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7759 - auc: 0.7957 - f1_score: 0.5607 - loss: 0.0839 - val_accuracy: 0.8509 - val_auc: 0.9051 - val_f1_score: 0.6574 - val_loss: 0.0627
Epoch 2/20
316/326 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8444 - auc: 0.8999 - f1_score: 0.6649 - loss: 0.0641Epoch 2 - Val F1: 0.6607, LR: 0.001000
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8445 - auc: 0.9000 - f1_score: 0.6650 - loss: 0.0641 - val_accuracy: 0.8530 - val_auc: 0.9057 - val_f1_score: 0.6607 - val_loss: 0.0625
Epoch 3/20
322/326 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8509 - auc: 0.9060 - f1_score: 0.6739 - loss: 0.0624Epoch 3 - Val F1: 0.6675, LR: 0.001000
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8509 - auc: 0.9060 - f1_score: 0.6740 - loss: 0.0624 - val_accuracy: 0.8522 - val_auc: 0.906

## Topic 3: Model Architecture Modification
Modifying architecture during training - PyTorch flexibility advantage

# Advanced Flexibility Showcase: PyTorch vs Keras Deep Dive

This section demonstrates **real-world flexibility differences** between PyTorch and Keras frameworks when implementing advanced machine learning techniques. We focus on **practical scenarios** where framework choice significantly impacts development workflow and capabilities.

## 🎯 Core Flexibility Areas Demonstrated

### Why Focal Loss Was Chosen

In our Adult Census Income example, we used **Focal Loss** specifically because of the **class imbalance** in the dataset:
- **76%** of samples have income ≤ 50K (majority class)
- **24%** of samples have income > 50K (minority class)

Standard Binary Cross-Entropy (BCE) loss would be dominated by the majority class, making the model poor at predicting high-income individuals. Focal Loss addresses this by **down-weighting well-classified examples** and **focusing training on hard examples**.

## Mathematical Formulation

### 1. **Binary Cross-Entropy (BCE) Foundation**

Focal Loss builds upon Binary Cross-Entropy:

$$\ell_{\text{BCE}}(p_t) = -\log(p_t)$$

Where:
- $p_t = \begin{cases}p & \text{if } y = 1 \\ 1-p & \text{if } y = 0\end{cases}$
- $p$ = predicted probability for class 1
- $y$ = true label (0 or 1)
- $p_t$ = probability of the true class

### 2. **Focal Loss Extension**

The complete Focal Loss function used in our example:

$$\ell_{\text{Focal}}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

**Parameters used in our code**:
- $\alpha = 0.75$ (higher weight for minority class)
- $\gamma = 2.0$ (focus parameter)

### 3. **Component Breakdown**

#### **a) Modulating Factor: $(1 - p_t)^\gamma$**
- When $p_t \rightarrow 1$ (well-classified example): $(1 - p_t)^\gamma \rightarrow 0$
  - **Effect**: Loss contribution approaches zero
- When $p_t \rightarrow 0$ (misclassified example): $(1 - p_t)^\gamma \rightarrow 1$
  - **Effect**: Full loss contribution preserved
- **$\gamma$ parameter**: Controls the rate of down-weighting
  - Higher $\gamma$ = stronger focus on hard examples
  - $\gamma = 0$ reduces to standard BCE

#### **b) Balancing Factor: $\alpha_t$**
- $\alpha_t = \begin{cases}\alpha & \text{if } y = 1 \\ 1-\alpha & \text{if } y = 0\end{cases}$
- **Purpose**: Addresses class imbalance
- **In our example**: $\alpha = 0.75$ gives higher weight to the minority class (income > 50K)
- **Why 0.75?**: Since minority class is ~24% of data, $\alpha = 0.75$ provides balanced weighting:
  - Majority class weight: $1 - 0.75 = 0.25$
  - Minority class weight: $0.75$
---
**PyTorch Implementation**:
```python
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        BCE_loss = nn.functional.binary_cross_entropy_with_logits(
            inputs, targets.float(), reduction='none'
        )
        pt = torch.exp(-BCE_loss)  # Probability of correct classification
        focal_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        return focal_loss.mean()
```

**Keras Implementation**:
```python
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        BCE = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        pt = tf.exp(-BCE)
        focal_loss = alpha * tf.pow(1-pt, gamma) * BCE
        return tf.reduce_mean(focal_loss)
    return loss_fn
```

**Flexibility Comparison**:
- ✅ **PyTorch**: Full OOP control with class-based implementation, easy parameter tuning
- ✅ **Keras**: Functional approach works well but limited debugging capabilities
- 🎯 **Key Difference**: PyTorch allows **dynamic parameter adjustment during training** (e.g., changing `alpha` based on epoch), while Keras requires recompiling the model

---

### 2. **Custom Metrics: F1 Score with Threshold Optimization**
**Problem**: Accuracy is misleading for imbalanced datasets. F1 score (harmonic mean of precision/recall) is better, but requires optimal threshold tuning.

**PyTorch Implementation**:
```python
class F1Score:
    def __init__(self, threshold=0.5):
        self.threshold = threshold
    
    def __call__(self, predictions, targets):
        pred_binary = (predictions > self.threshold).astype(np.float32)
        tp = np.sum(pred_binary * targets)
        fp = np.sum((1 - targets) * pred_binary)
        fn = np.sum(targets * (1 - pred_binary))
        
        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
        return f1
```

**Keras Implementation**:
```python
class F1ScoreMetric(tf.keras.metrics.Metric):
    def __init__(self, threshold=0.5, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')
    
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_binary = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        
        tp = tf.reduce_sum(y_true * y_pred_binary)
        fp = tf.reduce_sum((1 - y_true) * y_pred_binary)
        fn = tf.reduce_sum(y_true * (1 - y_pred_binary))
        
        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)
    
    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + 1e-7)
        recall = self.true_positives / (self.true_positives + self.false_negatives + 1e-7)
        return 2 * (precision * recall) / (precision + recall + 1e-7)
```

**Flexibility Comparison**:
- ✅ **PyTorch**: Complete control over calculation logic, easy threshold adjustment
- ✅ **Keras**: Built-in state management works well but requires strict class inheritance
- 🎯 **Key Difference**: PyTorch allows **threshold optimization during training** (e.g., finding best threshold on validation set), while Keras metrics are typically fixed during training

---

### 3. **Advanced Training Loop Control**
**Problem**: Real-world training requires custom logic beyond standard `fit()` methods: gradient clipping, dynamic batch sizing, manual optimization steps.

#### **PyTorch: Full Manual Control**
```python
class AdvancedPyTorchTrainer:
    def train(self, train_loader, val_loader, epochs=15):
        for epoch in range(epochs):
            self.model.train()
            total_loss = 0
            
            # Dynamic batch size adjustment based on epoch
            current_batch_size = min(64 + epoch * 8, 128)
            
            for batch_x, batch_y in train_loader:
                # Manual gradient clipping
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                
                self.optimizer.zero_grad()
                outputs = self.model(batch_x).squeeze()
                loss = self.criterion(outputs, batch_y)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
```

#### **Keras: Custom Callback Limitations**
```python
class DynamicTrainingCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        # Dynamic batch size adjustment (NOT POSSIBLE in standard Keras)
        # This would require complete custom training loop
        pass
    
    def on_epoch_end(self, epoch, logs=None):
        # Manual learning rate scheduling
        current_lr = tf.keras.backend.get_value(self.model.optimizer.learning_rate)
        new_lr = current_lr * 0.5  # Example reduction
        tf.keras.backend.set_value(self.model.optimizer.learning_rate, new_lr)
```

**Flexibility Comparison**:
- ✅ **PyTorch**: **Complete control** over every training step:
  - Manual gradient clipping (`clip_grad_norm_`)
  - Dynamic batch size changes
  - Custom optimization schedules
  - Mixed precision training control
- ⚠️ **Keras**: **Limited by callback system**:
  - Cannot change batch size during training
  - Gradient clipping requires custom training loops
  - Complex logic requires abandoning `fit()` method
- 🎯 **Key Difference**: PyTorch provides **true research flexibility** - you can implement **any training algorithm** without framework limitations. Keras requires **workarounds or complete custom loops** for advanced scenarios.

---

### 4. **Dynamic Architecture Modification**
**Problem**: Research scenarios often require modifying model architecture during training (e.g., adding layers, changing connections).

#### **PyTorch: Real-time Architecture Changes**
```python
class DynamicDenseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.hidden_layers = nn.ModuleList([...])
    
    def add_layer(self, new_units=32, dropout_rate=0.1):
        """Dynamically add a new layer DURING training"""
        new_layer = nn.Sequential(
            nn.Linear(last_output_dim, new_units),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.hidden_layers.append(new_layer)
        self.output_layer = nn.Linear(new_units, 1)
        
        # Reinitialize optimizer for new parameters
        print(f"Added new layer with {new_units} units")
```

#### **Keras: Architecture is Fixed After Compilation**
```python
# Keras CANNOT modify architecture during training
# Workaround: Train, save weights, rebuild model, reload weights
def add_layer_keras(old_model, new_units=32):
    # 1. Save current weights
    weights = old_model.get_weights()
    
    # 2. Create new architecture
    new_model = keras.Sequential([
        *old_model.layers,  # Copy existing layers
        layers.Dense(new_units, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(1, activation='sigmoid')
    ])
    
    # 3. Recompile (loses optimizer state!)
    new_model.compile(optimizer='adam', loss='binary_crossentropy')
    
    # 4. Attempt to load weights (often fails due to shape mismatch)
    try:
        new_model.set_weights(weights)
    except:
        print("Weight transfer failed - architecture mismatch")
    
    return new_model
```

**Flexibility Comparison**:
- ✅ **PyTorch**: **Game-changing capability** - modify architecture **during training** without restarting:
  - Add/remove layers dynamically
  - Change layer connections
  - Modify activation functions
  - Adjust dropout rates on-the-fly
- ❌ **Keras**: **Architecture is frozen** after compilation:
  - Requires complete model reconstruction
  - Loses optimizer state and momentum
  - Weight transfer often fails due to shape changes
  - Training must restart from scratch
- 🎯 **Key Difference**: PyTorch supports **lifelong learning architectures** and **neural architecture search** experiments that are impossible in Keras without major workarounds.

---

## 📊 Flexibility Scorecard

| Flexibility Aspect | PyTorch | Keras | Winner |
|-------------------|---------|-------|---------|
| **Custom Loss Functions** | ✅ Full class-based control | ✅ Functional approach | ⚖️ Tie |
| **Custom Metrics** | ✅ Complete calculation control | ✅ Built-in state management | ⚖️ Tie |
| **Gradient Control** | ✅ Manual clipping, custom steps | ⚠️ Requires custom loops | 🏆 PyTorch |
| **Training Loop Logic** | ✅ Complete manual control | ⚠️ Limited by callbacks | 🏆 PyTorch |
| **Dynamic Architecture** | ✅ Modify during training | ❌ Requires reconstruction | 🏆 PyTorch |
| **Learning Rate Scheduling** | ✅ Manual + built-in | ✅ Excellent built-in | ⚖️ Tie |
| **Debugging Experience** | ✅ Python-native debugging | ⚠️ Callback black box | 🏆 PyTorch |
| **Development Speed** | ⚠️ More boilerplate | ✅ Rapid prototyping | 🏆 Keras |

## 💡 When Flexibility Matters Most

### **Choose PyTorch When:**
- 🧪 **Research & Novel Algorithms**: Implementing unpublished papers or novel architectures
- 🔧 **Custom Training Dynamics**: Gradient manipulation, custom optimization steps
- 🏗️ **Dynamic Architectures**: Neural architecture search, lifelong learning systems
- 🎚️ **Fine-grained Control**: Every training step needs custom logic
- 🤖 **Advanced Regularization**: Custom dropout variants, layer-wise regularization

### **Choose Keras When:**
- 🚀 **Rapid Prototyping**: Getting baseline models running quickly
- 🏭 **Production Deployment**: Stable, well-tested pipeline deployment
- 📊 **Standard Workflows**: Using established training patterns
- 👥 **Team Collaboration**: Clean, readable code for non-experts
- 🛠️ **Built-in Features**: Leveraging Keras' excellent callbacks and utilities

## 🔮 Key Insight

> **"PyTorch gives you the keys to the research lab, while Keras gives you a well-equipped workshop."**

The Adult Census Income dataset with its class imbalance, mixed features, and real-world complexity perfectly demonstrates why **PyTorch's flexibility is essential for advanced ML research**, while **Keras excels at production-ready implementations** of established techniques.

This flexibility showcase isn't just about technical capabilities—it's about **unlocking new research possibilities** and **solving problems that standard frameworks cannot handle**.

In [ ]:
# PyTorch: Dynamic Architecture Modification Example
class DynamicDenseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Dropout(0.3)
            ),
            nn.Sequential(
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2)
            )
        ])
        self.output_layer = nn.Linear(64, 1)

    def forward(self, x):
        for layer in self.hidden_layers:
            x = layer(x)
        return self.output_layer(x)

    def add_layer(self, new_units=32, dropout_rate=0.1):
        """Dynamically add a new layer during training"""
        last_output_dim = self.hidden_layers[-1][0].out_features

        new_layer = nn.Sequential(
            nn.Linear(last_output_dim, new_units),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )

        self.hidden_layers.append(new_layer)
        self.output_layer = nn.Linear(new_units, 1)

        # Reinitialize optimizer for new parameters
        print(f"Added new layer with {new_units} units")

    def remove_layer(self, layer_idx=-1):
        """Dynamically remove a layer"""
        if len(self.hidden_layers) > 1:
            removed = self.hidden_layers.pop(layer_idx)
            new_output_dim = self.hidden_layers[-1][0].out_features
            self.output_layer = nn.Linear(new_output_dim, 1)
            print(f"Removed layer, new output dim: {new_output_dim}")

# Demonstration of dynamic architecture modification
dynamic_model = DynamicDenseNet(input_dim)
optimizer = optim.Adam(dynamic_model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

print("Initial architecture:")
print(dynamic_model)

# Train for a few epochs with initial architecture
for epoch in range(3):
    dynamic_model.train()
    total_loss = 0

    for batch_x, batch_y in train_val_loader:
        optimizer.zero_grad()
        outputs = dynamic_model(batch_x).squeeze()
        loss = criterion(outputs, batch_y.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_val_loader)
    print(f"Initial Epoch {epoch+1}, Loss: {avg_loss:.4f}")

# Dynamically add a new layer during training!
dynamic_model.add_layer(new_units=48, dropout_rate=0.15)

# Need to reinitialize optimizer for new parameters
optimizer = optim.Adam(dynamic_model.parameters(), lr=0.0005)  # Lower LR for fine-tuning

print("\nArchitecture after modification:")
print(dynamic_model)

# Continue training with modified architecture
for epoch in range(3):
    dynamic_model.train()
    total_loss = 0

    for batch_x, batch_y in train_val_loader:
        optimizer.zero_grad()
        outputs = dynamic_model(batch_x).squeeze()
        loss = criterion(outputs, batch_y.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_val_loader)
    print(f"Modified Epoch {epoch+1}, Loss: {avg_loss:.4f}")

print("\n✅ PyTorch demonstrates true architecture flexibility - layers can be added/removed DURING training")
print("❌ Keras requires complete model reconstruction for architecture changes")

Initial architecture:
DynamicDenseNet(
  (hidden_layers): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=108, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
    )
    (1): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
    )
  )
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)
Initial Epoch 1, Loss: 0.3603
Initial Epoch 2, Loss: 0.3157
Initial Epoch 3, Loss: 0.3100
Added new layer with 48 units

Architecture after modification:
DynamicDenseNet(
  (hidden_layers): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=108, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
    )
    (1): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
    )
    (2): Sequential(
      (0): Linear(in_features=64, o

## Results Summary & Framework Comparison

| Aspect | PyTorch | Keras |
|--------|---------|-------|
| **Custom Loss Functions** | ✅ Full flexibility with Python classes | ✅ Good support via functions |
| **Custom Metrics** | ✅ Manual implementation with full control | ✅ Class-based metrics work well |
| **Training Loop Control** | ✅ Complete manual control (gradients, clipping, batch logic) | ⚠️ Limited by fit() API, needs custom training loops |
| **Dynamic Architecture** | ✅ **HUGE ADVANTAGE**: Modify layers during training | ❌ Requires complete model reconstruction |
| **Gradient Manipulation** | ✅ Manual clipping, custom optimization steps | ⚠️ Possible but more complex |
| **Learning Rate Scheduling** | ✅ Full manual control + built-in schedulers | ✅ Good built-in support |
| **Ease of Use** | ⚠️ More boilerplate code | ✅ Very concise and clean |

### Key Insights:

**PyTorch Advantages:**
- **True research flexibility**: Modify architecture, loss functions, and training dynamics on-the-fly
- **Full gradient control**: Manual gradient clipping, custom optimization steps
- **Dynamic computation graph**: Changes during runtime are natural
- **Better for novel algorithms**: When you need to break standard training patterns

**Keras Advantages:**
- **Rapid prototyping**: Get baseline models running quickly
- **Clean API**: Less boilerplate code for standard tasks
- **Built-in utilities**: Callbacks, metrics, and preprocessing are well-integrated
- **Production ready**: Better deployment ecosystem

### When to Choose Which:

**Choose PyTorch when:**
- You need to modify architecture during training (like our dynamic layer example)
- Implementing novel loss functions or optimization algorithms
- Research settings where flexibility trumps convenience
- You need fine-grained control over every training step

**Choose Keras when:**
- Rapid prototyping of standard architectures
- Production deployment where stability matters
- Using well-established training patterns
- You prefer concise, readable code over maximum flexibility

**For your specific needs (flexibility in algorithm modification): PyTorch is clearly superior** for the advanced use cases you mentioned!

## Next Steps & Learning Activities

You've seen the core differences! Now let's deepen your understanding:

**Activity 1**: Try modifying the PyTorch focal loss to include class weights automatically  
**Activity 2**: Implement gradient accumulation in the PyTorch training loop  
**Activity 3**: Create a Keras custom training loop that matches PyTorch's flexibility

**Question for you**: Based on this comparison, which framework would you choose for:
1. A production income prediction model that needs to be deployed quickly?
2. Researching a completely new type of neural network architecture?

**Ready to try one of these activities, or do you have questions about any specific flexibility aspect we covered?**